# Multi-view reconstruction and Meshing

## 0. Import

In [ ]:
%matplotlib inline

import openalea.phenomenal.data as phm_data
import openalea.phenomenal.display as phm_display
import openalea.phenomenal.object as phm_obj
import openalea.phenomenal.multi_view_reconstruction as phm_mvr
import openalea.phenomenal.mesh as phm_mesh
import openalea.phenomenal.display.notebook as phm_display_notebook
from openalea.phenotyping_data.fetch import fetch_all_data

## 1. Prerequisites

### 1.1 Load data

In [ ]:
plant_number = 2  # Available : 1, 2, 3, 4 or 5
data_dir = fetch_all_data(f"plant_{plant_number}")

In [ ]:
bin_images = phm_data.bin_images(data_dir)
calibrations = phm_data.calibrations(data_dir)

phm_display.show_images(
    list(bin_images["side"].values()) + list(bin_images["top"].values())
)

## 2. Multi-view reconstruction

### 2.1 Associate images and projection function

In [ ]:
image_views = dict()
for id_camera in bin_images:
    for angle in bin_images[id_camera]:
        name = f'{id_camera}_{angle}'
        projection = calibrations[id_camera].get_projection(angle)
        image_views[name] = phm_obj.ImageView(
                bin_images[id_camera][angle],
                projection
            )

### 2.2 Do multi-view reconstruction

In [ ]:
voxels_size = 16  # mm
error_tolerance = 0
voxel_grid = phm_mvr.reconstruction_3d(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side')
phm_display_notebook.show_voxel_grid(voxel_grid)

In [ ]:
voxel_grid.bounding_box()

### 2.3 Evaluate reprojection

In [ ]:
phm_mvr.reconstruction_metrics(voxel_grid, image_views)

### 2.4 Save / Load voxel grid

In [ ]:
voxel_grid.write(f"plant_{plant_number}_size_{voxels_size}.npz")

In [ ]:
voxel_grid = phm_obj.VoxelGrid.read(f"plant_{plant_number}_size_{voxels_size}.npz")

## 3.Meshing

In [ ]:
vertices, faces = phm_mesh.meshing(
    voxel_grid.to_image_3d(), reduction=0.90, smoothing_iteration=5)
phm_display_notebook.show_mesh(vertices, faces)